# Análisis exploratorio de variantes

> **Notebook versionado sin salidas, de forma deliberada.** Sus celdas se
> ejecutan sobre los artefactos de la capa GOLD de la ejecución canónica
> (`data/processed/`), que no se versionan por tamaño. Guardar salidas aquí
> significaría arrastrar cifras de una ejecución distinta de la citada en la
> memoria, que es exactamente el problema que este proyecto trata de evitar.
> Para regenerarlas: `make data` y después `make eda`, con Python 3.12.
>
> Las cifras citables del análisis exploratorio están en `docs/EDA_variantes.md`
> y en las figuras de `docs/figuras/`, ambas producidas por el pipeline.


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.config import processed_dir, interim_dir, load_config

cfg = load_config()
FIG = ROOT / "docs" / "figuras"; FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

train = pd.read_parquet(processed_dir() / "train.parquet")
test = pd.read_parquet(processed_dir() / "test.parquet")
print("train:", train.shape, "| test:", test.shape)
train.head()

## 1. Balance del target
Prevalencia de variantes patogénicas en cada *release*.


In [ ]:
bal = pd.DataFrame({
 "train (%s)" % cfg["data"]["clinvar_train_release"]: train["label"].value_counts(normalize=True),
 "test (%s)" % cfg["data"]["clinvar_test_release"]: test["label"].value_counts(normalize=True),
}).sort_index()
ax = bal.T.plot(kind="bar", stacked=True, color=["#4C72B0","#C44E52"], figsize=(6,3.5))
ax.set_ylabel("proporción"); ax.set_title("Balance del target por release")
ax.legend(["Benigna (0)","Patogénica (1)"], loc="center left", bbox_to_anchor=(1,0.5))
plt.tight_layout(); plt.savefig(FIG/"eda_target_balance.png", bbox_inches="tight"); plt.show()
print(bal.round(3))

## 2. Separabilidad de las features
Si las features informan sobre la patogenicidad, sus distribuciones deben diferir entre clases.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13,3.6))
def kde_by_label(ax, col, title, logx=False):
 for lab, color, name in [(0,"#4C72B0","Benigna"),(1,"#C44E52","Patogénica")]:
 s = train.loc[train.label==lab, col].dropna()
 if logx: s = np.log10(s.clip(lower=1e-7))
 ax.hist(s, bins=40, density=True, alpha=0.55, color=color, label=name)
 ax.set_title(title); ax.legend()
kde_by_label(axes[0], "cadd_phred", "CADD phred")
kde_by_label(axes[1], "revel_score", "REVEL (solo missense)")
kde_by_label(axes[2], "gnomad_af", "gnomAD AF (log10)", logx=True)
plt.tight_layout(); plt.savefig(FIG/"eda_feature_separability.png", bbox_inches="tight"); plt.show()

## 3. Consecuencia funcional frente a patogenicidad


In [ ]:
ct = pd.crosstab(train["consequence"], train["label"], normalize="index").sort_values(1)
ax = ct.plot(kind="barh", stacked=True, color=["#4C72B0","#C44E52"], figsize=(7,4))
ax.set_xlabel("proporción"); ax.set_title("Patogenicidad por tipo de consecuencia (train)")
ax.legend(["Benigna","Patogénica"], loc="center left", bbox_to_anchor=(1,0.5))
plt.tight_layout(); plt.savefig(FIG/"eda_consequence.png", bbox_inches="tight"); plt.show()
print(ct.round(3))

## 4. Deriva entre releases (2023-12 y 2025-06)
Comparación de la distribución de significancia clínica cruda. La deriva viene de las
variantes nuevas y de las VUS reclasificadas: es la señal que sostiene la monitorización.


In [ ]:
raw_tr = pd.read_parquet(interim_dir()/f"annotated_{cfg['data']['clinvar_train_release']}.parquet")
raw_te = pd.read_parquet(interim_dir()/f"annotated_{cfg['data']['clinvar_test_release']}.parquet")
dist = pd.DataFrame({
 "2023-12": raw_tr["clnsig"].value_counts(normalize=True),
 "2025-06": raw_te["clnsig"].value_counts(normalize=True),
}).fillna(0).sort_index()
ax = dist.plot(kind="bar", figsize=(8,3.6), color=["#8172B3","#CCB974"])
ax.set_ylabel("proporción"); ax.set_title("Distribución de CLNSIG por release (drift)")
plt.tight_layout(); plt.savefig(FIG/"eda_clnsig_drift.png", bbox_inches="tight"); plt.show()

# Reclasificaciones VUS → etiqueta resuelta entre releases (por clave de variante)
key=["chrom","pos","ref","alt"]
m = raw_tr[key+["clnsig"]].merge(raw_te[key+["clnsig"]], on=key, suffixes=("_old","_new"))
reclass = m[(m.clnsig_old=="Uncertain_significance") & (m.clnsig_new!="Uncertain_significance")]
print("Variantes compartidas:", len(m))
print("VUS reclasificadas 2023-12 → 2025-06:", len(reclass))
print("Variantes nuevas solo en 2025-06:", len(raw_te) - len(m))
print(reclass["clnsig_new"].value_counts())

## 5. Conclusiones

* Target binarizado a patogénica frente a benigna, con las VUS reservadas como conjunto de inferencia.
* CADD, REVEL y la frecuencia de gnomAD en escala logarítmica separan las clases, con solapamiento: hay señal aprendible sin que el problema sea trivial.
* Las consecuencias truncantes concentran patogenicidad; las sinónimas, benignidad.
* Hay deriva real entre *releases* -variantes nuevas y VUS reclasificadas-, que justifica la monitorización continua.
